In [ ]:
import logging
import sys

import hydra
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import stats
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import torch


sys.path.insert(0, "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/collab-goettgens-SFC/shared_utils")
from experiment_utils import get_forward_model
from data_utils import transform_validation_data


In [ ]:
with hydra.initialize(
    config_path="../../inverse/loss_guidance/config"
):
    config = hydra.compose(
        config_name="run_inverse_constrained",
        overrides=[
            "paths=bloodplus_fm",
            "annotation=bloodplus",
            "sampling.N=10",
            "sampling.num_time_steps=20",
            "forward_model.num_time_steps=20",
            "constraints=default_all_cols"
        ]
    )
scatter_columns = config.annotation.scatter_columns
protocol_columns = config.annotation.protocol_columns

In [ ]:
logger = logging.getLogger(__name__)

forward_model, (
    flow_matching,
    target_prediction_model
) = get_forward_model(config, logger=logger)

In [ ]:
# Prepare label encoder
ct_le = LabelEncoder()
ct_values = target_prediction_model.train_data.adata.obs["cell_type"].values
ct_le.fit(ct_values)
classes = np.array(ct_le.classes_.tolist())


In [ ]:
adata_path_500k = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/data/gdrive/loop3/BloodPlus_Loop3_logicle_500k.h5ad"
adata_500k = sc.read_h5ad(adata_path_500k)
adata_500k


In [ ]:
cell_repr = "X_channel_standardized+X_scatter_standardized"
train_adata_g = target_prediction_model.train_data.adata
adata_500k = transform_validation_data(
    scatter_columns,
    train_adata_g,
    adata_500k,
    logger=None,
)
X_500k = adata_500k.obsm[cell_repr]


In [ ]:
ct_labels_list = []
chunck_size = 50_000
for chunck_idx in range(0, X_500k.shape[0], chunck_size):
    x_batch = X_500k[chunck_idx:chunck_idx+chunck_size]
    x_batch = torch.from_numpy(
        x_batch
    ).to(torch.float).to(target_prediction_model.device)

    with torch.no_grad():
        ct_logits = target_prediction_model.predict(x_batch)["cell_type"]
    ct_probs = torch.nn.functional.softmax(ct_logits, dim=1)
    ct_label = ct_probs.argmax(1).detach().cpu().numpy()
    ct_pred = ct_le.inverse_transform(ct_label)

    ct_labels_list.append(ct_pred)

ct_pred_500k = np.concatenate(ct_labels_list, axis=0)
adata_500k.obs["pred_cell_type"] = ct_pred_500k


In [ ]:
# compute neighbors and umap
sc.pp.neighbors(adata_500k)
sc.tl.umap(adata_500k)
adata_500k.obs["pred_cell_type"] = ct_pred_500k
sc.pl.embedding(
    adata_500k, "umap", s=20,
    color=["pred_cell_type", "FSC-A :: FSC - Area", "CD34", "CD41a", "CD14"],
    legend_loc="on data",
    frameon=False,
)